# Guide 11: Tool use and agents: letting a model act

*Lifecycle* · [Read the guide](https://drvgautam.github.io/primer-on-aiengineering/tools/) · [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drvgautam/primer-on-aiengineering/blob/main/code/notebooks/11-tools.ipynb)

This notebook holds every code block from Guide 11, in the guide's order: 11 code cells, 10 of which print an output the guide quotes. Each cell is preceded by the guide paragraph that introduces it, so you can keep the guide open beside the notebook. The saved outputs are the guide's; `tools/verify_notebooks.py` checks they still match.

Run the setup cell first, then the cells in order: like the guide's blocks, they share variables.

**Sections of the guide**

- Beyond text — no code
- The tool loop — 1 code cell
- Teaching a model to call tools — 4 code cells
- Facts that live outside the model — 2 code cells
- When the tool says something unexpected — 1 code cell
- Forcing the format: constrained decoding — 2 code cells
- Agents: tools in a loop — 1 code cell
- Bridge to retrieval — no code

In [1]:
# Setup: run this cell first. It makes the repository's code/ folder the working directory, so
# every block below runs exactly as printed in the guide ("run from the repository root").
import os, subprocess, sys
from pathlib import Path

REPO = "https://github.com/drvgautam/primer-on-aiengineering"
if "google.colab" in sys.modules and not any((p / "code" / "notebooks").is_dir() for p in [Path.cwd(), *Path.cwd().parents]):
    subprocess.run(["git", "clone", "--depth", "1", REPO, "/content/primer-on-aiengineering"], check=True)
    os.chdir("/content/primer-on-aiengineering")
    pins = [l.strip() for l in open("code/requirements.txt") if l.split("==")[0] in ("numpy", "tiktoken", "torch")]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pins,
                    "--extra-index-url", "https://download.pytorch.org/whl/cpu"], check=True)

here = Path.cwd().resolve()
code_dir = next(p / "code" for p in [here, *here.parents] if (p / "code" / "notebooks").is_dir())
os.chdir(code_dir)

# The guide's outputs were produced with these versions, on CPU. Others may differ in the last digits.
import numpy, tiktoken, torch
pins = dict(l.strip().split("==") for l in open("requirements.txt") if l.split("==")[0] in ("numpy", "tiktoken", "torch"))
for name, mod in (("numpy", numpy), ("tiktoken", tiktoken), ("torch", torch)):
    have = mod.__version__.split("+")[0]
    note = "" if have == pins[name] else f"  <- the guide used {pins[name]}; outputs may differ slightly (on Colab, restart the runtime and rerun this cell)"
    print(f"{name:9} {have}{note}")
print("working directory:", Path(code_dir.parent.name) / "code")

# %%gpt2 marks blocks the guide shows but doesn't run, because they need GPT-2 small's 548 MB weights.
# To run them, download the weights as series/running-example.py explains, then set:
#     os.environ["PRIMER_RUN_GPT2"] = "1"
from IPython.core.magic import register_cell_magic

@register_cell_magic
def gpt2(line, cell):
    if os.environ.get("PRIMER_RUN_GPT2") == "1":
        get_ipython().run_cell(cell)
    else:
        print("Skipped: needs GPT-2 weights (see the setup cell to enable).")


numpy     2.2.6
tiktoken  0.14.0
torch     2.14.1
working directory: primer-on-aiengineering/code


## The tool loop

The toy version uses only the toy alphabet of letters and spaces. The question is Guide 8's "where does the cat live ". The model writes "call home cat result", the application runs the toy's one tool, `home("cat")`, and appends " hut". The model then writes " in the hut", followed by " and". Guide 8's training answers all ended in " and", so it serves as the toy's stop sequence: the application stops generation there and drops the marker (real chat formats end a turn with a special token instead, as Guide 8 noted). Here is the application's side, the code that spots a finished call and runs the tool. Run the blocks from the repository root, so Python can find the series' modules in `guides/`:

In [2]:
import sys
sys.path.insert(0, "guides")                   # run from the repository root
import torch
from tools import (load_base, tool_example, sft_segments, run_agent, combinations, where_check,
                   sampled, val_loss, greedy, homes, nouns, vocab)

def intercept(text, table):                    # the application: if a call just ended, run the tool
    if text.endswith(" result"):
        animal = text.split()[-2]              # "... call home cat result" -> "cat"
        return " " + table.get(animal, "error")
    return ""

print(repr(intercept("where does the cat live call home cat result", homes)))

' hut'


## Teaching a model to call tools

A base model has never seen "call home cat result". Tool use is learned the way Guide 8 taught answering: supervised fine-tuning on examples, here examples that contain calls and results. Recall from Guide 8 that its data was single (prompt, response) pairs, with loss masking keeping the prompt out of the loss. A tool example has more pieces, and they're written by different authors:

In [3]:
example = tool_example("cat", "rug")           # a random place, not the cat's home
for text, trained in example:
    print(f"{text!r:28} {'loss' if trained else 'no loss'}")

'where does the cat live '   no loss
'call home cat result'       loss
' rug'                       no loss
' in the rug and'            loss


The model writes the call and the answer, so those get a loss. The prompt comes from the user and the result comes from the application, so they get none. Training the model to predict the tool result, though, would teach it to invent results instead of waiting for them. Each position's target is the next character, so the loss mask marks a position when the character it predicts was written by the model:

In [4]:
def segment_mask(segments):                    # 1 where the next character is model-written
    flags = [trained for text, trained in segments for _ in text]
    return torch.tensor(flags[1:], dtype=torch.float)   # one flag per position (characters - 1); position i predicts character i + 1

whole = "".join(text for text, _ in example)
print(whole[1:])
print("".join(str(int(f)) for f in segment_mask(example)))

here does the cat live call home cat result rug in the rug and
00000000000000000000000111111111111111111110000111111111111111


The run starts from Guide 8's base model and trains for 400 steps on batches of 32, using Adam (Guide 7's optimizer) at a learning rate of 0.001. Each batch is half tool questions, each with a random animal and a random place as the result, and half Guide 8's 100 "where" questions, like "the fox hid under the rug where " answered by "under the rug", which need no tool. Those teach the model when *not* to call. Then the application asks about the cat:

In [5]:
model = sft_segments(load_base(), segment_mask, seed=0)   # 400 steps, under 15 s on a laptop CPU
print(run_agent(model, "where does the cat live ", homes, intercept))

call home cat result hut in the hut


That line is the whole transcript after the question: the model's call, the application's " hut", and the model's answer. To check that the model copies rather than remembers, the next block tries every animal with every place as the tool's result, 36 combinations. It also asks the 115 held-out "where" questions, which shouldn't trigger a call at all:

In [6]:
print(f"{combinations(model, intercept)} of 36 animal and place combinations right")
no_call, right = where_check(model, intercept)
print(f"held-out where questions: no call in {no_call} of 115, exactly right {right}")

36 of 36 animal and place combinations right


held-out where questions: no call in 115 of 115, exactly right 107


## Facts that live outside the model

Guide 8 couldn't teach the toy that the cat moved without breaking every other answer. With a tool, the cat moves to the pot by changing one entry in the table:

In [7]:
moved = dict(homes, cat="pot")                 # the cat moves: change one entry in the tool's table
for a in nouns:
    print(a, run_agent(model, f"where does the {a} live ", moved, intercept))

cat call home cat result pot in the pot
dog call home dog result bed in the bed
rat call home rat result log in the log
fox call home fox result rug in the rug
owl call home owl result pot in the pot


hen call home hen result mat in the mat


This fine-tune still had a cost, the same one Guide 8 found. Recall that fine-tuning on a narrow task makes a model forget some of what pre-training taught it, measured by its loss on held-out sentences from the toy world:

In [8]:
print(f"validation loss {val_loss(load_base()):.3f} before, {val_loss(model):.3f} after")
print(repr(greedy(model, "the cat lives in the ", 3)))

validation loss 0.244 before, 0.859 after
'owl'


## When the tool says something unexpected

The model learned to copy places it had seen in training. What does it do with a result it never saw? Here the tool returns an animal, and two words that never appear in the toy world:

In [9]:
for result in ("dog", "mud", "sun"):
    print(result, run_agent(model, "where does the cat live ", {"cat": result}, intercept))

dog call home cat result dog in the log
mud call home cat result mud in the bed
sun call home cat result sun the pot in the log 


## Forcing the format: constrained decoding

So far the decoding rule was greedy, always taking the top-scoring character from the model's distribution. Applications often sample at a temperature above zero to get varied answers, and that's where call formats break: sample at a higher temperature and the toy starts writing malformed calls, which the application can't parse. Recall from Guide 6 that top-k sampling keeps the $k$ most probable tokens and gives every other token probability zero. **Constrained decoding** makes the same move with a different rule: before sampling, give zero probability to every token the required format doesn't allow next. The code does it with a constraint mask added to the logits, 0 for allowed characters and $-\infty$ for the rest, since $e^{-\infty}$ is zero after the softmax; recall that Guide 4's causal mask used the same trick to hide later positions. Here the format is "call home *animal* result", and the allowed characters are the ones that keep the text on the way to one of the six valid calls:

In [10]:
calls = [f"call home {a} result" for a in nouns]

def constrain(logits, written):                # logits: len(vocab) = 27 scores for the next character; written: the model's text after the prompt, before any result
    allowed = {c[len(written)] for c in calls if c.startswith(written) and len(c) > len(written)}
    mask = torch.full_like(logits, float("-inf"))
    mask[[vocab.index(ch) for ch in allowed]] = 0
    return logits + mask

The application applies the constraint mask only until the call has run; the answer after the result is sampled freely. This block asks 60 tool questions per setting, 10 per animal, sampled with a fixed seed, at three temperatures, with and without the constraint:

In [11]:
for T in (1.0, 1.5, 2.0):
    for rule in (None, constrain):
        formed, named, right = sampled(model, intercept, rule, T)
        print(f"T={T}  {'constrained  ' if rule else 'unconstrained'}  well-formed {formed:2}"
              f"  right animal {named:2}  right answer {right:2}")

T=1.0  unconstrained  well-formed 58  right animal 56  right answer 54


T=1.0  constrained    well-formed 60  right animal 57  right answer 55


T=1.5  unconstrained  well-formed 33  right animal 29  right answer 25


T=1.5  constrained    well-formed 60  right animal 51  right answer 40


T=2.0  unconstrained  well-formed  5  right animal  4  right answer  2


T=2.0  constrained    well-formed 60  right animal 51  right answer 21


## Agents: tools in a loop

Every step is a chance to go wrong: a malformed call, the wrong tool, a result misread like the toy's "mud". Suppose each step succeeds with probability $s$, independently of the others. A task needs every step to succeed, so its chance of success is $s$ multiplied by itself once per step, $s$ raised to the number of steps:

In [12]:
for s in (0.99, 0.95, 0.90):
    print(f"s = {s:.2f}:", "  ".join(f"{s ** steps:6.1%}" for steps in (5, 10, 20)))

s = 0.99:  95.1%   90.4%   81.8%
s = 0.95:  77.4%   59.9%   35.8%
s = 0.90:  59.0%   34.9%   12.2%


---

← [Guide 10: Inference and serving](10-inference.ipynb)

Found a cell that doesn't reproduce the guide? [Open an issue](https://github.com/drvgautam/primer-on-aiengineering/issues/new?template=does-not-reproduce.yml).